In [1]:
import seaborn as sns 
import pandas as pd 
from sklearn.model_selection import train_test_split 
from sklearn.ensemble import RandomForestRegressor 
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score 
import optuna 
from sklearn.model_selection import cross_val_score 
import matplotlib.pyplot as plt

/Users/beto_valdes/opt/anaconda3/envs/optuna_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
healthexp = sns.load_dataset('healthexp') 
healthexp.head(100)

,Year,Country,Spending_USD,Life_Expectancy
0,1970,Germany,252.311,70.6
1,1970,France,192.143,72.2
2,1970,Great Britain,123.993,71.9
3,1970,Japan,150.437,72.0
4,1970,USA,326.961,70.9
...,...,...,...,...
95,1991,Canada,1805.209,77.6
96,1991,France,1558.033,77.2
97,1991,Great Britain,842.797,75.9
98,1991,Japan,1166.430,79.1


In [3]:
healthexp = pd.get_dummies(healthexp)

In [4]:
X = healthexp.drop(['Life_Expectancy'], axis=1)

In [5]:
y = healthexp['Life_Expectancy']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=19)

In [7]:
rfr = RandomForestRegressor(random_state=13)

In [8]:
rfr.fit(X_train, y_train)

RandomForestRegressor(random_state=13)

In [9]:
y_pred = rfr.predict(X_test)

In [10]:
mean_absolute_error(y_test, y_pred)

0.2592727272727093

In [11]:
mean_squared_error(y_test, y_pred)

0.10287989090908921

In [12]:
r2_score(y_test, y_pred)

0.990987198652996

In [13]:
def objective(trial): 
    n_estimators = trial.suggest_int('n_estimators', 100, 1000) 
    max_depth = trial.suggest_int('max_depth', 10, 50) 
    min_samples_split = trial.suggest_int('min_samples_split', 2, 32)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 32)
    model = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf) 
    score = cross_val_score(model, X, y, n_jobs=-1, cv=5, scoring='neg_mean_absolute_error').mean()
    return score

In [14]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler(seed=42)) # Default is random Search

[I 2025-05-27 19:06:19,618] A new study created in memory with name: no-name-ddc3c460-2435-4bab-9588-3a450a85753d


In [15]:
study.optimize(objective, n_trials=100)

[I 2025-05-27 19:06:21,107] Trial 0 finished with value: -1.6632067868212388 and parameters: {'n_estimators': 437, 'max_depth': 48, 'min_samples_split': 24, 'min_samples_leaf': 20}. Best is trial 0 with value: -1.6632067868212388.
[I 2025-05-27 19:06:21,984] Trial 1 finished with value: -1.7977716884935737 and parameters: {'n_estimators': 240, 'max_depth': 16, 'min_samples_split': 3, 'min_samples_leaf': 28}. Best is trial 0 with value: -1.6632067868212388.
[I 2025-05-27 19:06:22,969] Trial 2 finished with value: -1.8765785326664168 and parameters: {'n_estimators': 641, 'max_depth': 39, 'min_samples_split': 2, 'min_samples_leaf': 32}. Best is trial 0 with value: -1.6632067868212388.
[I 2025-05-27 19:06:24,176] Trial 3 finished with value: -1.3219310044178658 and parameters: {'n_estimators': 850, 'max_depth': 18, 'min_samples_split': 7, 'min_samples_leaf': 6}. Best is trial 3 with value: -1.3219310044178658.
[I 2025-05-27 19:06:24,478] Trial 4 finished with value: -1.5354627065517306 and

In [16]:
best_params = study.best_params 
print(f"Best Hyperparameters: {best_params}")

Best Hyperparameters: {'n_estimators': 358, 'max_depth': 34, 'min_samples_split': 2, 'min_samples_leaf': 2}


In [17]:
best_score = study.best_value
print(f"Best Accuracy: {best_score:.3f}")

Best Accuracy: -1.013


In [18]:
optuna.visualization.plot_optimization_history(study)

In [19]:
optuna.visualization.plot_parallel_coordinate(study)

In [20]:
optuna.visualization.plot_slice(study, params=['n_estimators', 'max_depth', 'min_samples_leaf', 'min_samples_split'])

In [21]:
optuna.visualization.plot_param_importances(study)

In [22]:
best_n_estimators = best_params['n_estimators']
best_max_depth = best_params['max_depth']
best_min_samples_split = best_params['min_samples_split']
best_min_samples_leaf = best_params['min_samples_leaf']

In [23]:
best_model = RandomForestRegressor(n_estimators=best_n_estimators, max_depth=best_max_depth, min_samples_split=best_min_samples_split, min_samples_leaf=best_min_samples_leaf)
best_model.fit(X_train, y_train)

RandomForestRegressor(max_depth=34, min_samples_leaf=2, n_estimators=358)

In [24]:
y_pred = best_model.predict(X_test)

In [25]:
mean_absolute_error(y_test, y_pred)

0.31296957128420655

In [26]:
mean_squared_error(y_test, y_pred)

0.13933496437504425

In [27]:
r2_score(y_test, y_pred)

0.9877935489286837